# 02 — Encoder Probing: Phase 2a Evaluation\n\n**Generated**: 2026-06-06  \n**Phase**: Phase 2a — Modality Encoders  \n**Data**: 7 persona archetypes, ~1000 records per modality\n\n## Purpose\n\nEvaluate all 4 modality encoders on strategy recovery and embedding quality.\nEach encoder is probed via frozen encoder + logistic regression on a stratified train/test split.\n\n## Probe Results Summary\n\n| Encoder | Strategy Recovery | Threshold | Status |\n|---|---|---|---|\n| Text | 99.0% ± 0.3% | >70% | ✅ |\n| Psychographic | 100.0% ± 0.0% | >75% | ✅ |\n| Trace | 37.8% ± 0.7% | >85% | ❌ |\n| Transaction | N/A (training bug) | >60% | ❌ |\n\n## Text Encoder Details\n- Intra-persona cosine sim: 0.71 (>0.6 ✅)\n- Inter-persona cosine sim: -0.03 (<0.4 ✅)\n- Sentence-transformer + linear projection works well\n\n## Key Findings\n1. Data has 7 participants (one per persona), not 1000 independent participants\n2. Supervised encoders achieve near-perfect accuracy (expected with 7 separable classes)\n3. Contrastive trace encoder underperforms (37.75%) due to limited class diversity\n4. Transaction encoder has training loop bug (double backward) — deferred to Phase 2b\n5. MLP encoder does not improve over raw 22-dim features (both 100% accuracy)

In [ ]:
# Run all probes and regenerate plots\nimport os\nos.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'\n!cd .. && PYTHONPATH=. uv run python -m evaluation.run_probes 2>&1 | grep -E '(strategy recovery|PASS|FAIL|cosine)'

In [ ]:
# Generate UMAP and evaluation plots\n!cd .. && PYTHONPATH=. uv run python evaluation/generate_probe_plots.py

## UMAP Projections\n\n*(Generated by evaluation/generate_probe_plots.py)*\n\n![Text Encoder UMAP](text_umap.png)\n![Psychographic Encoder UMAP](psycho_umap.png)\n![Trace Encoder UMAP](trace_umap.png)

## Strategy Recovery Comparison\n\n![Strategy Recovery Bars](strategy_recovery_bars.png)

## Confusion Matrices\n\n![Text Confusion Matrix](text_confusion.png)\n![Psychographic Confusion Matrix](psycho_confusion.png)

## Cosine Similarity Heatmap (Text Encoder)\n\n![Cosine Similarity Heatmap](text_cosine_sim_heatmap.png)\n\n---\n\n## Human Review Gate 🛑\n\n**STOP**: Review the UMAP projections, confusion matrices, and strategy recovery\nbar chart before proceeding to Phase 2b (Fusion). Key questions:\n\n1. Do embeddings form visually separable clusters by persona?\n2. Are the confusion patterns consistent with expected persona similarities?\n   (e.g., price_lex vs quality_lex should be the hardest to separate)\n3. Is the trace encoder's 37.75% recovery acceptable, or does the training\n   objective need redesign before Phase 2b?\n4. Does the transaction encoder bug need fixing before fusion?\n\n**Decision**: [ ] Proceed to Phase 2b  /  [ ] Fix issues first